In [24]:
!pip install pytest
!pip install playwright
!playwright install



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [25]:
# ============================================================
# 1. Imports
# ============================================================

from pathlib import Path
import subprocess
from typing import Dict, List


# ============================================================
# 2. Load + Parse Natural-Language Test Case
# ============================================================

def load_test_case(path: str) -> str:
    return Path(path).read_text(encoding="utf-8")


def parse_test_case(text: str) -> Dict:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    goal = ""
    base_url = ""
    steps: List[str] = []
    expected = ""

    in_steps = False
    in_expected = False

    for line in lines:
        lower = line.lower()

        if lower.startswith("goal:"):
            goal = line.split(":", 1)[1].strip()

        elif lower.startswith("url:"):
            base_url = line.split(":", 1)[1].strip()

        elif lower.startswith("steps"):
            in_steps = True
            in_expected = False

        elif lower.startswith("expected result"):
            in_expected = True
            in_steps = False

        elif in_steps:
            if line[0].isdigit() and "." in line:
                _, rest = line.split(".", 1)
                steps.append(rest.strip())
            else:
                steps.append(line.strip())

        elif in_expected:
            expected += line + " "

    return {
        "goal": goal,
        "base_url": base_url,
        "raw_steps": steps,
        "expected": expected.strip(),
    }


# ============================================================
# 3. Interpret Steps → IR
# ============================================================

def interpret_step(step: str) -> Dict:
    s = step.lower()

    if s.startswith("navigate to"):
        target = step.split("to", 1)[1].strip()
        return {"action": "navigate", "url": target}

    if s.startswith("enter username"):
        value = step.split("username", 1)[1].strip().strip('"')
        return {"action": "enter_text", "selector": "username_input", "value": value}

    if s.startswith("enter password"):
        value = step.split("password", 1)[1].strip().strip('"')
        return {"action": "enter_text", "selector": "password_input", "value": value}

    if s.startswith("click login button"):
        return {"action": "click", "selector": "login_button"}

    if s.startswith("verify"):
        parts = step.split()
        logical = parts[1] if len(parts) >= 2 else "unknown"
        return {"action": "assert_visible", "selector": logical}

    return {"action": "noop", "raw": step}


def build_ir(parsed: Dict) -> Dict:
    return {
        "goal": parsed["goal"],
        "base_url": parsed["base_url"],
        "expected": parsed["expected"],
        "steps": [interpret_step(s) for s in parsed["raw_steps"]],
    }


# ============================================================
# 4. Selector Mapping
# ============================================================

SELECTOR_MAP = {
    "username_input": 'input[name="username"]',
    "password_input": 'input[name="password"]',
    "login_button": 'button[type="submit"]',
    "dashboard_heading": "h1",
}


def map_ir(ir: Dict) -> Dict:
    mapped = []
    for step in ir["steps"]:
        new_step = dict(step)
        logical = new_step.get("selector")
        if logical in SELECTOR_MAP:
            new_step["selector"] = SELECTOR_MAP[logical]
        mapped.append(new_step)

    return {
        "goal": ir["goal"],
        "base_url": ir["base_url"],
        "expected": ir["expected"],
        "steps": mapped,
    }


# ============================================================
# 5. Deterministic Playwright Codegen
# ============================================================


TEST_TEMPLATE = """import pytest
from playwright.sync_api import sync_playwright

def test_login_flow():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False, slow_mo=300)
        page = browser.new_page()
{body}
        browser.close()
"""



def _indent(line: str, level: int = 2) -> str:
    return " " * (4 * level) + line

def generate_test_body(ir: Dict) -> str:
    lines = []
    base_url = ir["base_url"].rstrip("/")

    for step in ir["steps"]:
        action = step["action"]

        # -----------------------------
        # NAVIGATION LOGIC (FIXED)
        # -----------------------------
        if action == "navigate":
            url = step["url"].strip()

            if url.startswith("http"):
                # Full URL provided in test case
                final_url = url

            elif url.startswith("/"):
                # Avoid double-appending /login when base_url already ends with it
                if base_url.endswith(url):
                    final_url = base_url
                else:
                    final_url = base_url + url

            else:
                # Relative path like "login"
                final_url = base_url + "/" + url

            lines.append(_indent(f'page.goto("{final_url}")'))
            continue

        # -----------------------------
        # ENTER TEXT (FIXED ESCAPING)
        # -----------------------------
        if action == "enter_text":
            sel = step["selector"]
            val = step["value"]

            # Escape quotes so Python doesn't break
            safe_sel = sel.replace('"', '\\"')
            safe_val = val.replace('"', '\\"')

            lines.append(_indent(f'page.fill("{safe_sel}", "{safe_val}")'))
            continue

        # -----------------------------
        # CLICK
        # -----------------------------
        if action == "click":
            sel = step["selector"]
            safe_sel = sel.replace('"', '\\"')
            lines.append(_indent(f'page.click("{safe_sel}")'))
            continue

        # -----------------------------
        # ASSERT VISIBLE
        # -----------------------------
        if action == "assert_visible":
            sel = step["selector"]
            safe_sel = sel.replace('"', '\\"')
            lines.append(_indent(f'page.wait_for_selector("{safe_sel}")'))
            continue

        # -----------------------------
        # NOOP
        # -----------------------------
        lines.append(_indent(f"# noop: {step}"))

    return "\n".join(lines)


def generate_test_code(ir: Dict, output_path: str) -> str:
    body = generate_test_body(ir)
    code = TEST_TEMPLATE.format(body=body)
    Path(output_path).write_text(code, encoding="utf-8")
    return output_path


# ============================================================
# 6. Run Playwright Test
# ============================================================

def run_test_file(test_path: str):
    cmd = ["pytest", "-q", test_path]
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    out, _ = proc.communicate()
    return proc.returncode, out


# ============================================================
# 7. Full Pipeline Runner
# ============================================================

def run_pipeline():
    base = Path().resolve()

    # FIXED PATH — notebook is inside /notebooks, test file is one level up
    sample = (base / ".." / "sample-data" / "test_cases" / "test_login_success.txt").resolve()

    artifacts = base / "artifacts" / "good"
    artifacts.mkdir(parents=True, exist_ok=True)

    generated_test = base / "pipeline_test_login.py"

    # 1. Load + parse
    text = load_test_case(str(sample))
    parsed = parse_test_case(text)

    # 2. IR
    ir = build_ir(parsed)

    # 3. Selector mapping
    mapped_ir = map_ir(ir)

    # 4. Generate deterministic test
    generate_test_code(mapped_ir, str(generated_test))

    # 5. Run test
    code, output = run_test_file(str(generated_test))

    # 6. Save output
    (artifacts / ("pass_output.txt" if code == 0 else "fail_output.txt")).write_text(
        output, encoding="utf-8"
    )

    print("Exit code:", code)
    print(output)


# ============================================================
# 8. Execute Pipeline
# ============================================================

run_pipeline()


Exit code: 0
.                                                                        [100%]
1 passed in 24.81s



In [26]:
# ============================================================
# LLM‑Only Pipeline (Expected to FAIL)
# ============================================================

from pathlib import Path
import subprocess
import textwrap
import random


# ------------------------------------------------------------
# 1. Load the same NL test case
# ------------------------------------------------------------

def load_test_case(path: str) -> str:
    return Path(path).read_text(encoding="utf-8")


base = Path().resolve()
sample = (base / ".." / "sample-data" / "test_cases" / "test_login_success.txt").resolve()
nl_test = load_test_case(str(sample))


# ------------------------------------------------------------
# 2. LLM Simulator — realistic hallucinated Playwright generator
# ------------------------------------------------------------

def simulate_llm_generated_test(nl_test: str) -> str:
    """
    Produces Playwright code that looks like an LLM wrote it,
    including realistic hallucinations and selector mistakes.
    """

    # Common LLM hallucinated selectors
    username_guesses = [
        "#username-input", "#user", "input#user-name", "input[name='user']"
    ]
    password_guesses = [
        "#password-field", "#pass", "input#pwd", "input[name='pass']"
    ]
    button_guesses = [
        "#submit-login", "button.login-btn", "button[type='login']", ".btn-primary"
    ]
    heading_guesses = [
        "#dashboard-title", "h2.dashboard-title", "h1.page-title", ".dashboard-header"
    ]

    return textwrap.dedent(f"""
    import pytest
    from playwright.sync_api import sync_playwright
    import time

    def test_login_flow_llm_brittle():
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=False, slow_mo=300)
            page = browser.new_page()

            # Force a short timeout so the browser closes quickly
            page.set_default_timeout(2000)  # 2 seconds per step

            start = time.time()
            try:
                # LLM-generated (simulated) code — intentionally brittle
                page.goto("https://ai-test-automation-demo.onrender.com/login")

                page.fill("{random.choice(username_guesses)}", "demo")
                page.fill("{random.choice(password_guesses)}", "password123")
                page.click("{random.choice(button_guesses)}")
                page.wait_for_selector("{random.choice(heading_guesses)}")

            finally:
                # Ensure browser closes after ~7 seconds max
                elapsed = time.time() - start
                remaining = max(0, 7 - elapsed)
                time.sleep(remaining)
                browser.close()
    """)


# ------------------------------------------------------------
# 3. Generate and write the brittle test file
# ------------------------------------------------------------

hallucinated_code = simulate_llm_generated_test(nl_test)

brittle_test_path = base / "pipeline_test_login_llm_brittle.py"
brittle_test_path.write_text(hallucinated_code, encoding="utf-8")


# ------------------------------------------------------------
# 4. Run the brittle test
# ------------------------------------------------------------

def run_test_file(test_path: str):
    cmd = ["pytest", "-q", test_path]
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    out, _ = proc.communicate()
    return proc.returncode, out


code, output = run_test_file(str(brittle_test_path))

# ------------------------------------------------------------
# 5. Save artifacts
# ------------------------------------------------------------

artifacts = base / "artifacts" / "brittle"
artifacts.mkdir(parents=True, exist_ok=True)

(artifacts / ("pass_output.txt" if code == 0 else "fail_output.txt")).write_text(
    output, encoding="utf-8"
)

print("Exit code:", code)
print(output)


Exit code: 1
F                                                                        [100%]
=================================== FAILURES ===================================
_________________________ test_login_flow_llm_brittle __________________________

    def test_login_flow_llm_brittle():
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=False, slow_mo=300)
            page = browser.new_page()
    
            # Force a short timeout so the browser closes quickly
            page.set_default_timeout(2000)  # 2 seconds per step
    
            start = time.time()
            try:
                # LLM-generated (simulated) code — intentionally brittle
                page.goto("https://ai-test-automation-demo.onrender.com/login")
    
>               page.fill("#user", "demo")

pipeline_test_login_llm_brittle.py:19: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 
../.venv/lib/python3.11/site-packages/playwright

Tell a GenAI tool to do below, copy and pase.

Write a Playwright test for this natural language test case:

Goal: Login Success Test
URL: https://ai-test-automation-demo.onrender.com/login

Steps:
1. Navigate to /login
2. Enter username "demo"
3. Enter password "password123"
4. Click login button
5. Verify dashboard_heading visible

Expected Result:
User is redirected to the dashboard and sees the welcome message.

IMPORTANT:
- Use **raw Playwright**, not pytest-playwright.
- Launch the browser manually using:

    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        page = browser.new_page()

- Do NOT use pytest fixtures like `page: Page`.
- Do NOT use `expect()`.
- Do NOT use `page.get_by_role()` or `page.get_by_label()`.
- Use only:
    page.goto()
    page.fill()
    page.click()
    page.wait_for_selector()

- Close the browser manually with browser.close().


In [27]:
# ============================================================
# Real LLM Output Pipeline (You Paste the Code)
# ============================================================

from pathlib import Path
import subprocess
import textwrap

# ------------------------------------------------------------
# 1. Paste LLM-generated Playwright code here
# ------------------------------------------------------------
# Go to ChatGPT / Claude / Gemini / Groq
# Paste your NL test case
# Ask: "Generate a Playwright test for this"
# Then paste ONLY the Playwright test code below.

llm_generated_code = """
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)
    page = browser.new_page()

    # 1. Navigate to /login
    page.goto("https://ai-test-automation-demo.onrender.com/login")

    # 2. Enter username "demo"
    page.fill('input[name="username"]', "demo")

    # 3. Enter password "password123"
    page.fill('input[name="password"]', "password123")

    # 4. Click login button
    page.click('button[type="submit"]')

    # 5. Verify dashboard_heading visible
    page.wait_for_selector('[data-testid="dashboard_heading"]')

    print("Login Success Test Passed")

    browser.close()
"""

# ------------------------------------------------------------
# 2. Write the pasted code to a test file
# ------------------------------------------------------------

base = Path().resolve()
real_llm_test_path = base / "pipeline_test_login_llm_real.py"
real_llm_test_path.write_text(llm_generated_code, encoding="utf-8")


# ------------------------------------------------------------
# 3. Run the real LLM-generated test
# ------------------------------------------------------------

def run_test_file(test_path: str):
    cmd = ["pytest", "-q", test_path]
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    out, _ = proc.communicate()
    return proc.returncode, out


code, output = run_test_file(str(real_llm_test_path))

# ------------------------------------------------------------
# 4. Save artifacts
# ------------------------------------------------------------

artifacts = base / "artifacts" / "real_llm"
artifacts.mkdir(parents=True, exist_ok=True)

(artifacts / ("pass_output.txt" if code == 0 else "fail_output.txt")).write_text(
    output, encoding="utf-8"
)

print("Exit code:", code)
print(output)


Exit code: 2

==================================== ERRORS ====================================
_______________ ERROR collecting pipeline_test_login_llm_real.py _______________
pipeline_test_login_llm_real.py:21: in <module>
    page.wait_for_selector('[data-testid="dashboard_heading"]')
../.venv/lib/python3.11/site-packages/playwright/sync_api/_generated.py:8770: in wait_for_selector
    self._sync(
../.venv/lib/python3.11/site-packages/playwright/_impl/_page.py:424: in wait_for_selector
    return await self._main_frame.wait_for_selector(**locals_to_params(locals()))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
../.venv/lib/python3.11/site-packages/playwright/_impl/_frame.py:372: in wait_for_selector
    await self._channel.send(
../.venv/lib/python3.11/site-packages/playwright/_impl/_connection.py:69: in send
    return await self._connection.wrap_api_call(
../.venv/lib/python3.11/site-packages/playwright/_impl/_connection.py:559: in wrap_api_call